In [0]:
from pyspark.sql import functions as F

# 1. Qual é a receita total (em R$) somada de todos os filmes da base?

In [0]:
#Somando a receita em reais dos filmes que estão na tabela fato. A soma ignora receitas nulas, assim, não transformo dado ausente em receita zero
df_receita_total = spark.table("workspace.gold.fact_movies_performance").agg(
    F.round(F.sum("receita_brl"), 2).alias("receita_total_brl"))

display(df_receita_total)

#  2. Quais são os 5 filmes com maior popularidade? Mostre título e valor de popularidade.

In [0]:
# Unindo as métricas ao título e ordenando pela popularidade
df_mais_populares = (
    spark.table("workspace.gold.fact_movies_performance").alias("f")
    .join(
        spark.table("workspace.gold.dim_movies").alias("m"),
        on="sk_movie_id",
        how="inner"
    )
    .select("titulo", "popularidade")
    .orderBy(F.col("popularidade").desc())
    .limit(5))

display(df_mais_populares)

#  3. Quantos filmes cada gênero possui? Liste do maior para o menor volume.

In [0]:
#Contando os filmes associados em cada gênero
df_filmes_por_genero = (
    spark.table("workspace.gold.bridge_movie_genre")
    .join(
        spark.table("workspace.gold.dim_genres"),
        on="sk_genre_id",
        how="inner"
    )
    .groupBy("nome_genero")
#Utilizei countDistinct para não contar duas vezes o mesmo filme dentro do gênero
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.col("qtd_filmes").desc()))

display(df_filmes_por_genero)

# 4. Para os 10 filmes de maior receita, mostre título, receita (em US$ e R$) e a posição de cada um no ranking
(RANK()).

In [0]:
from pyspark.sql.window import Window

In [0]:
#Unindo receita e título e calculo a posição no ranking de receita em dólares
janela_receita = Window.orderBy(F.col("receita_usd").desc())

df_top_receitas = (
    spark.table("workspace.gold.fact_movies_performance").alias("f")
    .join(
        spark.table("workspace.gold.dim_movies").alias("m"),
        on="sk_movie_id",
        how="inner"
    )
#RANK atribui a mesma posição quando dois filmes têm a mesma receita
    .withColumn("posicao_ranking", F.rank().over(janela_receita))
    .select("titulo", "receita_usd", "receita_brl", "posicao_ranking")
    .orderBy("posicao_ranking")
    .limit(10))

display(df_top_receitas)

# 5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos*?

In [0]:
#Definindo a data limite usando o lançamento mais recente que já aconteceu
df_filmes = spark.table("workspace.gold.dim_movies")

#O período termina no lançamento realizado mais recente, excluo lançamentos futuros e status diferentes de lançado
data_limite = (
    df_filmes
    .filter(
        (F.col("status_filme") == "Lançado")
        & (F.col("data_lancamento") <= F.current_date())
    )
    .agg(F.max("data_lancamento").alias("data_limite"))
    .first()["data_limite"])

data_inicio = F.add_months(F.lit(data_limite), -24)

#Contando em quantos filmes cada ator participou dentro desse período
df_atores_periodo = (
    spark.table("workspace.gold.bridge_movie_person").alias("b")
    .join(
        spark.table("workspace.gold.dim_people").alias("p"),
        on="sk_person_id",
        how="inner"
    )
    .join(
        df_filmes.alias("m"),
        on="sk_movie_id",
        how="inner"
    )
    .filter(
        (F.col("tipo_pessoa") == "Ator")
        & (F.col("status_filme") == "Lançado")
        & (F.col("data_lancamento") >= data_inicio)
        & (F.col("data_lancamento") <= F.lit(data_limite))
    )
    .groupBy("nome_pessoa")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes")))

#Caso haja empate, é mostrado todos os atores com a maior quantidade
janela_atores = Window.orderBy(F.col("qtd_filmes").desc())

df_ator_com_mais_filmes = (
    df_atores_periodo
    .withColumn("posicao", F.rank().over(janela_atores))
    .filter(F.col("posicao") == 1)
    .drop("posicao"))

display(df_ator_com_mais_filmes)

# 6. Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos*?

In [0]:
#Usando o último lançamento válido como limite superior do período
df_filmes = spark.table("workspace.gold.dim_movies")

df_filmes_validos = df_filmes.filter(
    (F.col("status_filme") == "Lançado")
    & (F.col("data_lancamento") <= F.current_date()))

data_limite = (
    df_filmes_validos
    .agg(F.max("data_lancamento").alias("data_limite"))
    .first()["data_limite"])

data_inicio = F.add_months(F.lit(data_limite), -60)

#Somando o lucro dos filmes do período para cada produtora
df_lucro_produtoras = (
    spark.table("workspace.gold.bridge_movie_company")
    .join(
        spark.table("workspace.gold.dim_companies"),
        on="sk_company_id",
        how="inner"
    )
    .join(
        spark.table("workspace.gold.fact_movies_performance"),
        on="sk_movie_id",
        how="inner"
    )
    .join(
        df_filmes_validos.select("sk_movie_id", "data_lancamento"),
        on="sk_movie_id",
        how="inner"
    )
    .filter(
        (F.col("data_lancamento") >= data_inicio)
        & (F.col("data_lancamento") <= F.lit(data_limite))
    )
    .groupBy("nome_produtora")
    .agg(
        F.sum("lucro_usd").alias("lucro_total_usd"),
        F.sum("lucro_brl").alias("lucro_total_brl")
    )
    .orderBy(F.col("lucro_total_brl").desc())
    .limit(1))

display(df_lucro_produtoras)